In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
# 1. 데이터 경로 확인
import os
import pandas as pd

DATA_DIR = "/kaggle/input/korea-food"  
print("존재함?", os.path.exists(DATA_DIR))
print("하위 폴더:", os.listdir(DATA_DIR)[:20])

존재함? True
하위 폴더: ['고등어구이', '족발', '계란말이', '양념치킨', '감자탕', '김밥']


In [15]:
# 2 ImageFolder로 클래스/이미지 수 확인
from torchvision import datasets
base_dataset = datasets.ImageFolder(root=DATA_DIR)
print("클래스:", base_dataset.classes)
print("클래스 수:", len(base_dataset.classes))
print("전체 이미지 수:", len(base_dataset))

클래스: ['감자탕', '계란말이', '고등어구이', '김밥', '양념치킨', '족발']
클래스 수: 6
전체 이미지 수: 4433


In [16]:
# 3. 폴더에서 파일 리스트 + 라벨 만들기
class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
class_to_idx = {name: i for i, name in enumerate(class_names)}
c_to_idx = {val : key for  key, val in class_to_idx.items()}
print("c_to_idx:", class_to_idx)

c_to_idx: {'감자탕': 0, '계란말이': 1, '고등어구이': 2, '김밥': 3, '양념치킨': 4, '족발': 5}


In [17]:
# (이미지 경로, 클래스명, target) rows 만들기
rows = []
for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(".jpg"):
            rows.append({
                "filepath": os.path.join(cls_dir, fname),  # 전체 경로
                "label": cls,                              # 문자 라벨
                "target": class_to_idx[cls]                # 정수 라벨
            })

In [19]:
label_df = pd.DataFrame(rows)
label_df

,filepath,label,target
0,/kaggle/input/korea-food/감자탕/Img_135_0638.jpg,감자탕,0
1,/kaggle/input/korea-food/감자탕/Img_135_0850.jpg,감자탕,0
2,/kaggle/input/korea-food/감자탕/Img_135_0555.jpg,감자탕,0
3,/kaggle/input/korea-food/감자탕/Img_135_0964.jpg,감자탕,0
4,/kaggle/input/korea-food/감자탕/Img_135_0463.jpg,감자탕,0
...,...,...,...
4397,/kaggle/input/korea-food/족발/Img_131_0165.jpg,족발,5
4398,/kaggle/input/korea-food/족발/Img_131_0260.jpg,족발,5
4399,/kaggle/input/korea-food/족발/Img_131_0890.jpg,족발,5
4400,/kaggle/input/korea-food/족발/Img_131_0143.jpg,족발,5


In [20]:
# 4. Train_test_split
from sklearn.model_selection import train_test_split

y = label_df["target"]
X = label_df[["filepath"]] 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
# 5. Dataset - 음식 이미 filepath 가지고 있으니깐 그걸로 열면 됨
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class FoodDataset(Dataset):
    def __init__(self, df, y, transform=None):
        self.df = df.reset_index(drop=True)
        self.y = y.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "filepath"]
        image = Image.open(img_path).convert("RGB")
        label = int(self.y.loc[idx])

        if self.transform:
            image = self.transform(image)

        return image, label

In [22]:
# 6. Transform 준비 (아직 증강 아님 - 전이 들어가기 전 최소 전처리)
basic_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [26]:
# 7. DataLoader 
train_dataset = FoodDataset(X_train, y_train, transform=basic_tf)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = FoodDataset(X_test, y_test, transform=basic_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
# -- 훈련
dataloaders = {"train": train_loader, "val": test_loader}

In [39]:
## 8.전이학습 단계! VGG16
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import VGG16_Weights
from tqdm import tqdm

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 6
BATCH_SIZE = 32  
NUM_EPOCHS = 10
MODE = "feature_extraction"      

class VGG16TransferLearning(nn.Module):
    def __init__(self, num_classes: int, mode: str = "feature_extraction"):
        super().__init__()

        self.backbone = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        
        if mode == "feature_extraction":
            for param in self.backbone.features.parameters():
                param.requires_grad = False
        elif mode == 'fine_tuning':
            freeze_until = 24  
            for idx, param in enumerate(self.backbone.features.parameters()):
                if idx < freeze_until:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
                    print(f"Fine-tuning enabled at layer index: {idx}")
       
        in_features = self.backbone.classifier[6].in_features
        self.backbone.classifier[6] = nn.Linear(in_features, num_classes)

        
        nn.init.kaiming_normal_(self.backbone.classifier[6].weight, mode="fan_out", nonlinearity="relu")
        nn.init.constant_(self.backbone.classifier[6].bias, 0)

    def forward(self, x):
        return self.backbone(x)

model = VGG16TransferLearning(num_classes=NUM_CLASSES, mode=MODE).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / Total: {total:,} ({trainable/total:.2%})")

Trainable params: 119,570,438 / Total: 134,285,126 (89.04%)


In [40]:
# 9. Loss/Optimizer 설정
criterion = nn.CrossEntropyLoss()

import torch.optim as optim
if MODE == 'feature_extraction' :
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),  
        lr=1e-4,
        weight_decay=1e-4
    )
else :
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,        
        momentum=0.9,   
        weight_decay=5e-4
    )

In [41]:
# 10. train 모델 그대로 사용
def train_model(model, dataloaders, criterion, optimizer,
                num_epochs=25, device='cuda'):
    """
    전이학습 학습 루프
    Args:
        model: VGG16TransferLearning 인스턴스
        dataloaders: {'train': DataLoader, 'val': DataLoader}
        criterion: 손실 함수 (CrossEntropyLoss)
        optimizer: optimizer (SGD with momentum 권장)
    """
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 60)

        # 각 epoch마다 train -> val phase 순환
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Dropout 활성화, BN 학습 모드
            else:
                model.eval()   # Dropout 비활성화, BN 추론 모드

            running_loss = 0.0
            running_corrects = 0

            # 진행률 표시
            pbar = tqdm(dataloaders[phase], desc=phase)

            for inputs, labels in pbar:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                # gradient 누적 초기화
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)  # 예측 클래스
                    loss = criterion(outputs, labels)

                    # Backward + Optimize (train phase only)
                    if phase == 'train':
                        loss.backward()

                        # Gradient Clipping (VGG16은 깊어서 안정성을 위해 권장)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                        optimizer.step()

                # 통계 계산
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                # tqdm 업데이트
                pbar.set_postfix({'loss': loss.item()})

            # Epoch 통계
            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 최적 모델 저장 (validation 기준)
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict().copy()
                torch.save(best_model_wts, 'best_vgg16_transfer.pth')

    print(f'\nBest val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

In [42]:
# 11. 학습 루프 실행
dataloaders = {'train' : train_loader , 'val' : test_loader}
model, history = train_model(
    model, dataloaders, criterion, optimizer,
    num_epochs=NUM_EPOCHS, device=device
)


Epoch 1/10
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:37<00:00,  2.98it/s, loss=0]    


train Loss: 5.8663 Acc: 0.7265


val: 100%|██████████| 28/28 [00:10<00:00,  2.66it/s, loss=1.71]  


val Loss: 0.9836 Acc: 0.8763

Epoch 2/10
------------------------------------------------------------


...

val Loss: 3.1933 Acc: 0.9296

Epoch 10/10
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:38<00:00,  2.92it/s, loss=0]       


train Loss: 0.7012 Acc: 0.9756


val: 100%|██████████| 28/28 [00:10<00:00,  2.57it/s, loss=12.2]   

val Loss: 5.7860 Acc: 0.8933

Best val Acc: 0.9342


In [43]:
## - 이미지 증강
# 1. train/val transforms
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [45]:
# 2. Dataset / DataLoader를 “증강용”으로 다시 만들기
train_dataset = FoodDataset(X_train, y_train, transform=train_transforms)
val_dataset   = FoodDataset(X_test,  y_test,  transform=val_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

dataloaders = {"train": train_loader, "val": val_loader}

In [46]:
model, history_aug = train_model(
    model, dataloaders, criterion, optimizer,
    num_epochs=NUM_EPOCHS, device=device
)


Epoch 1/10
------------------------------------------------------------


train:  47%|████▋     | 52/111 [00:18<00:19,  3.04it/s, loss=4.23]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
train: 100%|██████████| 111/111 [00:40<00:00,  2.76it/s, loss=0.000977]


train Loss: 4.5438 Acc: 0.7532


val: 100%|██████████| 28/28 [00:11<00:00,  2.54it/s, loss=1.1]   


val Loss: 0.8484 Acc: 0.8865

Epoch 2/10
------------------------------------------------------------


...

val Loss: 0.2807 Acc: 0.9308

Epoch 10/10
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:40<00:00,  2.77it/s, loss=0.305]


train Loss: 0.6208 Acc: 0.8160


val: 100%|██████████| 28/28 [00:11<00:00,  2.51it/s, loss=0.332] 

val Loss: 0.3023 Acc: 0.9274

Best val Acc: 0.9308


In [47]:
# 불균형 확인
counts_all = label_df["target"].value_counts().sort_index()

print("=== 전체 클래스별 개수 (target 기준) ===")
print(counts_all)

print("\n비율(%)")
print((counts_all / counts_all.sum() * 100).round(2))

=== 전체 클래스별 개수 (target 기준) ===
target
0     742
1     203
2    1000
3     982
4     922
5     553
Name: count, dtype: int64

비율(%)
target
0    16.86
1     4.61
2    22.72
3    22.31
4    20.95
5    12.56
Name: count, dtype: float64


In [49]:
# 가중치 부여 ㅜ 
import torch
NUM_CLASSES = 6  

train_counts = y_train.value_counts().sort_index() 
counts = torch.zeros(NUM_CLASSES, dtype=torch.float)

for i in range(NUM_CLASSES):
    counts[i] = float(train_counts.get(i, 0))

print("train class counts:", counts.tolist())

train class counts: [594.0, 162.0, 800.0, 785.0, 738.0, 442.0]


In [50]:
counts = torch.clamp(counts, min=1.0)
class_weights = 1.0 / counts
class_weights = class_weights / class_weights.mean()

print("class weights:", class_weights.tolist())

criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

class weights: [0.7216201424598694, 2.6459405422210693, 0.535802960395813, 0.5460412502288818, 0.5808162093162537, 0.9697791337966919]


In [51]:
model, history = train_model(
    model, dataloaders, criterion, optimizer,
    num_epochs=NUM_EPOCHS, device=device
)


Epoch 1/10
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:40<00:00,  2.73it/s, loss=0]    


train Loss: 0.6929 Acc: 0.8324


val: 100%|██████████| 28/28 [00:11<00:00,  2.51it/s, loss=0.263] 


val Loss: 0.2734 Acc: 0.9194

Epoch 2/10
------------------------------------------------------------


...

val Loss: 0.1993 Acc: 0.9398

Epoch 10/10
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:40<00:00,  2.72it/s, loss=0.812]


train Loss: 0.5895 Acc: 0.8390


val: 100%|██████████| 28/28 [00:11<00:00,  2.49it/s, loss=0.174] 

val Loss: 0.2548 Acc: 0.9330

Best val Acc: 0.9410


In [53]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.eval()

all_preds = []
all_labels = []

loader = val_loader  # 또는 test_loader / dataloaders["val"]

with torch.no_grad():
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

print(classification_report(
    y_true, y_pred,
    target_names=class_names if "class_names" in globals() else None,
    digits=4
))

              precision    recall  f1-score   support

         감자탕     0.8944    0.9730    0.9320       148
        계란말이     0.8810    0.9024    0.8916        41
       고등어구이     0.9602    0.9650    0.9626       200
          김밥     0.9783    0.9137    0.9449       197
        양념치킨     0.9419    0.8804    0.9101       184
          족발     0.8760    0.9550    0.9138       111

    accuracy                         0.9330       881
   macro avg     0.9220    0.9316    0.9258       881
weighted avg     0.9351    0.9330    0.9331       881



In [54]:
# -- fine-tuning
# 1. fine-tuning 모델 생성
MODE = "fine_tuning"
NUM_CLASSES = 6  # len(class_names)로 해도 됨
FREEZE_UNTIL = 24

model = VGG16TransferLearning(num_classes=NUM_CLASSES, mode=MODE)  # 네 클래스 정의 그대로 사용
model = model.to(device)

Fine-tuning enabled at layer index: 24
Fine-tuning enabled at layer index: 25


In [55]:
# 2.feature_extraciton에서 저장한 best weight 불러오기
ckpt_path = "best_vgg16_transfer.pth"  
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state)
print("Loaded checkpoint:", ckpt_path)

Loaded checkpoint: best_vgg16_transfer.pth


In [56]:
# 3. Loss - weighted 그대로 유지 -> 불균형이라 가중치 넣었기 때문
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

In [57]:
# 4.Optimizer -> SGD + 작은 LR로
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,        
    momentum=0.9,
    weight_decay=5e-4
)

In [58]:
# 5. fine-tuning 학습 실행
FT_EPOCHS = 12

model, history_ft = train_model(
    model, dataloaders, criterion, optimizer,
    num_epochs=FT_EPOCHS, device=device
)


Epoch 1/12
------------------------------------------------------------


train:   0%|          | 0/111 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
train: 100%|██████████| 111/111 [00:40<00:00,  2.73it/s, loss=0]    


train Loss: 0.5660 Acc: 0.8375


val: 100%|██████████| 28/28 [00:10<00:00,  2.58it/s, loss=0.252] 


val Loss: 0.2163 Acc: 0.9432

Epoch 2/12
------------------------------------------------------------


...

val Loss: 0.2116 Acc: 0.9410

Epoch 12/12
------------------------------------------------------------


train: 100%|██████████| 111/111 [00:40<00:00,  2.71it/s, loss=0.0979]


train Loss: 0.4073 Acc: 0.8713


val: 100%|██████████| 28/28 [00:10<00:00,  2.55it/s, loss=0.335] 

val Loss: 0.2253 Acc: 0.9376

Best val Acc: 0.9444


In [59]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.eval()

all_preds = []
all_labels = []

loader = val_loader  # 또는 test_loader / dataloaders["val"]

with torch.no_grad():
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

print(classification_report(
    y_true, y_pred,
    target_names=class_names if "class_names" in globals() else None,
    digits=4
))

              precision    recall  f1-score   support

         감자탕     0.9221    0.9595    0.9404       148
        계란말이     0.9268    0.9268    0.9268        41
       고등어구이     0.9369    0.9650    0.9507       200
          김밥     0.9775    0.8832    0.9280       197
        양념치킨     0.9402    0.9402    0.9402       184
          족발     0.8983    0.9550    0.9258       111

    accuracy                         0.9376       881
   macro avg     0.9336    0.9383    0.9353       881
weighted avg     0.9389    0.9376    0.9375       881



In [62]:
torch.save(model.state_dict(), "./best_food_vgg16_ft.pth")

In [63]:
import pickle

with open("label.pkl", "wb") as f:
    pickle.dump({"label": class_names}, f)

print("saved label.pkl:", class_names)

saved label.pkl: ['감자탕', '계란말이', '고등어구이', '김밥', '양념치킨', '족발']
